# GPU Training for Parkinson's Disease Detection

This notebook downloads the dataset from the repo (or uses the already‑committed CSV), builds a tiny VGG‑style model, trains it on a GPU, and saves the trained checkpoint back to the `models/` folder of the repository.

In [ ]:
# Install dependencies
!pip -q install tensorflow pandas scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd, numpy as np, os
# Path to the dataset inside the repo (mounted automatically when we clone)
data_path = '/content/parkinsons-disease-detection/data/parkinsons_updrs.csv'
df = pd.read_csv(data_path)
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# Binary label (total_UPDRS > 15)
df['label'] = (df['total_UPDRS'] > 15).astype(int)
X = df.drop(columns=['subject#','test_time','total_UPDRS','label']).values.astype(float)
y = df['label'].values
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
print('Train shape', X_train.shape, 'Val shape', X_val.shape)

In [ ]:
import tensorflow as tf
def reshape_features(x):
    x = tf.expand_dims(x, -1)
    return tf.reshape(x, (-1, 2, 11, 1))
inputs = tf.keras.Input(shape=(X_train.shape[1],), name='features')
x = tf.keras.layers.Lambda(reshape_features)(inputs)
x = tf.keras.layers.Conv2D(32,(3,3),padding='same')(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.MaxPooling2D((2,2))(x)
x = tf.keras.layers.Conv2D(64,(3,3),padding='same')(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(64,activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1,activation='sigmoid')(x)
model = tf.keras.Model(inputs,outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                          loss='binary_crossentropy',
                          metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
               epochs=30, batch_size=32, verbose=2)
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend()
plt.show()

In [ ]:
# Save model into the repo's models folder
model_path = '/content/parkinsons-disease-detection/models/best_trained.keras'
model.save(model_path)
print('Model saved to', model_path)

In [ ]:
# Download the checkpoint so you can push it back to GitHub
from google.colab import files
files.download(model_path)